In [1073]:
import pandas as pd
df = pd.read_csv("data/processed/player_match_stats.csv")
print(df.shape)
df.head()


(27909, 25)


,match_id,player,team,opposition,venue,city,date,season,toss_winner,toss_decision,...,batting_position,wickets,runs_conceded,balls_bowled,maidens,economy,total_wickets,player_innings,team_total,total_fantasy_points
0,1426261,TM Head,Sunrisers Hyderabad,Punjab Kings,Maharaja Yadavindra Singh International Cricke...,Mohali,2024-04-09,2024,Punjab Kings,field,...,1,0,0,0,0,0.0,6,1,182,31
1,1426261,Abhishek Sharma,Sunrisers Hyderabad,Punjab Kings,Maharaja Yadavindra Singh International Cricke...,Mohali,2024-04-09,2024,Punjab Kings,field,...,2,0,0,0,0,0.0,6,1,182,26
2,1426261,AK Markram,Sunrisers Hyderabad,Punjab Kings,Maharaja Yadavindra Singh International Cricke...,Mohali,2024-04-09,2024,Punjab Kings,field,...,3,0,0,0,0,0.0,6,1,182,4
3,1426261,Nithish Kumar Reddy,Sunrisers Hyderabad,Punjab Kings,Maharaja Yadavindra Singh International Cricke...,Mohali,2024-04-09,2024,Punjab Kings,field,...,4,1,33,18,0,11.0,6,1,182,114
4,1426261,RA Tripathi,Sunrisers Hyderabad,Punjab Kings,Maharaja Yadavindra Singh International Cricke...,Mohali,2024-04-09,2024,Punjab Kings,field,...,5,0,0,0,0,0.0,6,1,182,14


In [1074]:
avg_bat_position = df.groupby("player")["batting_position"].agg(lambda x: x.mode()[0])

In [1075]:
print(avg_bat_position)

player
A Ashish Reddy    7
A Badoni          6
A Chandila        0
A Chopra          2
A Choudhary       0
                 ..
Younis Khan       3
Yudhvir Singh     0
Yuvraj Singh      4
Z Khan            0
Zeeshan Ansari    0
Name: batting_position, Length: 811, dtype: int64


In [1076]:
def bin_batting_position(pos):
    if pos == 0:
        return 0  # didn't bat
    elif pos <= 3:
        return 1  # top order
    elif pos <= 6:
        return 2  # middle order
    else:
        return 3  # lower order

df["batting_position_bucket"] = df["batting_position"].apply(bin_batting_position)

In [1077]:
df["batting_position_bucket"].isna()

0        False
1        False
2        False
3        False
4        False
         ...  
27904    False
27905    False
27906    False
27907    False
27908    False
Name: batting_position_bucket, Length: 27909, dtype: bool

In [1078]:
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["player","date"]).reset_index(drop = True)
df[["player","date","total_fantasy_points"]].head(20)

,player,date,total_fantasy_points
0,A Ashish Reddy,2012-04-26,28
1,A Ashish Reddy,2012-04-29,32
2,A Ashish Reddy,2012-05-01,10
3,A Ashish Reddy,2012-05-04,19
4,A Ashish Reddy,2012-05-06,10
5,A Ashish Reddy,2012-05-08,31
6,A Ashish Reddy,2012-05-10,-8
7,A Ashish Reddy,2012-05-18,16
8,A Ashish Reddy,2012-05-20,46
9,A Ashish Reddy,2013-04-05,18


In [1079]:
df["Total_career_runs"] = df.groupby(["player"])["runs"].transform(lambda x : x.shift(1).expanding().sum())
df["Total_career_wickets"] = df.groupby(["player"])["wickets"].transform(lambda x : x.shift(1).expanding().sum())


In [1080]:
df[df["player"] == "V Kohli"][["player", "date", "runs", "wickets","Total_career_runs" , "Total_career_wickets"]].head(10)
df[df["player"] == "JJ Bumrah"][["player", "date", "runs", "wickets", "Total_career_runs", "Total_career_wickets"]]


,player,date,runs,wickets,Total_career_runs,Total_career_wickets
9352,JJ Bumrah,2013-04-04,0,3,NaN,NaN
9353,JJ Bumrah,2013-04-21,0,0,0.0,3.0
9354,JJ Bumrah,2014-04-19,1,0,0.0,3.0
9355,JJ Bumrah,2014-05-03,0,0,1.0,3.0
9356,JJ Bumrah,2014-05-06,0,2,1.0,3.0
...,...,...,...,...,...,...
9505,JJ Bumrah,2026-05-02,0,1,75.0,188.0
9506,JJ Bumrah,2026-05-04,0,0,75.0,189.0
9507,JJ Bumrah,2026-05-10,0,0,75.0,189.0
9508,JJ Bumrah,2026-05-14,0,0,75.0,189.0


In [1081]:
df[["player","Total_career_runs" , "Total_career_wickets"]]

,player,Total_career_runs,Total_career_wickets
0,A Ashish Reddy,NaN,NaN
1,A Ashish Reddy,0.0,2.0
2,A Ashish Reddy,10.0,3.0
3,A Ashish Reddy,10.0,4.0
4,A Ashish Reddy,13.0,5.0
...,...,...,...
27904,Zeeshan Ansari,0.0,4.0
27905,Zeeshan Ansari,0.0,5.0
27906,Zeeshan Ansari,0.0,5.0
27907,Zeeshan Ansari,0.0,6.0


In [1082]:
df["rolling_avg_fantasy_5"] = df.groupby("player")["total_fantasy_points"].transform(lambda x: x.shift(1).rolling(5, min_periods = 1).mean())
df["rolling_std_fantasy_5"] = df.groupby("player")["total_fantasy_points"].transform(lambda x: x.shift(1).rolling(5, min_periods = 1).std())
df[["player", "date", "total_fantasy_points", "rolling_avg_fantasy_5","rolling_std_fantasy_5"]].head(20)

,player,date,total_fantasy_points,rolling_avg_fantasy_5,rolling_std_fantasy_5
0,A Ashish Reddy,2012-04-26,28,NaN,NaN
1,A Ashish Reddy,2012-04-29,32,28.000000,NaN
2,A Ashish Reddy,2012-05-01,10,30.000000,2.828427
3,A Ashish Reddy,2012-05-04,19,23.333333,11.718931
4,A Ashish Reddy,2012-05-06,10,22.250000,9.810708
5,A Ashish Reddy,2012-05-08,31,19.800000,10.109402
6,A Ashish Reddy,2012-05-10,-8,20.400000,10.784248
7,A Ashish Reddy,2012-05-18,16,12.400000,14.293355
8,A Ashish Reddy,2012-05-20,46,13.600000,14.293355
9,A Ashish Reddy,2013-04-05,18,19.000000,20.566964


In [1083]:
df[df["player"] == "V Kohli"][["player", "date", "total_fantasy_points", "rolling_avg_fantasy_5","rolling_std_fantasy_5"]]

,player,date,total_fantasy_points,rolling_avg_fantasy_5,rolling_std_fantasy_5
25844,V Kohli,2008-04-18,5,NaN,NaN
25845,V Kohli,2008-04-20,36,5.0,NaN
25846,V Kohli,2008-04-26,19,20.5,21.920310
25847,V Kohli,2008-04-28,18,20.0,15.524175
25848,V Kohli,2008-04-30,-1,19.5,12.714821
...,...,...,...,...,...
26121,V Kohli,2026-05-13,162,43.2,50.375589
26122,V Kohli,2026-05-17,96,50.2,65.185888
26123,V Kohli,2026-05-22,23,62.2,67.403264
26124,V Kohli,2026-05-26,60,57.8,69.492446


In [1084]:
df["rolling_avg_fantasy_3"] = df.groupby("player")["total_fantasy_points"].transform(lambda x: x.shift(1).rolling(3,min_periods= 1).mean())

df["rolling_std_fantasy_10"] = df.groupby("player")["total_fantasy_points"].transform(lambda x: x.shift(1).rolling(5,min_periods = 1).std())

df["rolling_avg_fantasy_10"] = df.groupby("player")["total_fantasy_points"].transform(lambda x: x.shift(1).rolling(10,min_periods= 1).mean())

df["rolling_avg_runs_5"] = df.groupby("player")["runs"].transform(lambda x: x.shift(1).rolling(5,min_periods = 1).mean())

df["rolling_avg_wickets_5"] = df.groupby("player")["wickets"].transform(lambda x: x.shift(1).rolling(5,min_periods = 1).mean())

df["matches_played"] = df.groupby("player")["date"].transform(lambda x: x.expanding().count().shift(1).fillna(0))


print("Form features added")
df[["player", "date", "total_fantasy_points", "rolling_avg_fantasy_3", 
    "rolling_avg_fantasy_5", "rolling_avg_fantasy_10", 
    "rolling_avg_runs_5", "rolling_avg_wickets_5", 
    "matches_played","rolling_std_fantasy_10"]].head(20)

Form features added


,player,date,total_fantasy_points,rolling_avg_fantasy_3,rolling_avg_fantasy_5,rolling_avg_fantasy_10,rolling_avg_runs_5,rolling_avg_wickets_5,matches_played,rolling_std_fantasy_10
0,A Ashish Reddy,2012-04-26,28,NaN,NaN,NaN,NaN,NaN,0.0,NaN
1,A Ashish Reddy,2012-04-29,32,28.000000,28.000000,28.000000,0.000000,2.000000,1.0,NaN
2,A Ashish Reddy,2012-05-01,10,30.000000,30.000000,30.000000,5.000000,1.500000,2.0,2.828427
3,A Ashish Reddy,2012-05-04,19,23.333333,23.333333,23.333333,3.333333,1.333333,3.0,11.718931
4,A Ashish Reddy,2012-05-06,10,20.333333,22.250000,22.250000,3.250000,1.250000,4.0,9.810708
5,A Ashish Reddy,2012-05-08,31,13.000000,19.800000,19.800000,2.600000,1.200000,5.0,10.109402
6,A Ashish Reddy,2012-05-10,-8,20.000000,20.400000,21.666667,4.200000,1.200000,6.0,10.784248
7,A Ashish Reddy,2012-05-18,16,11.000000,12.400000,17.428571,2.200000,1.000000,7.0,14.293355
8,A Ashish Reddy,2012-05-20,46,13.000000,13.600000,17.250000,4.200000,0.800000,8.0,14.293355
9,A Ashish Reddy,2013-04-05,18,18.000000,19.000000,20.444444,4.400000,1.200000,9.0,20.566964


In [1085]:
df["expanding_season_fantasy_avg"] = df.groupby(["player" , "season"])["total_fantasy_points"].transform( lambda x: x.shift(1).expanding().mean())
df["expanding_season_fantasy_std"] = df.groupby(["player", "season"])["total_fantasy_points"].transform(lambda x : x.shift(1).expanding().std())

In [1086]:
df["batting_contribution"] = df.apply(lambda row: row["runs"]/row["team_total"] if row["team_total"] > 0 else 0, axis = 1)
df["bowling_contribution"] = df.apply(lambda row: row["wickets"]/row["total_wickets"] if row["total_wickets"] > 0 else 0, axis = 1)
print(df[["player", "runs", "team_total", "batting_contribution", "wickets", "total_wickets", "bowling_contribution"]].head(10))

           player  runs  team_total  batting_contribution  wickets  \
0  A Ashish Reddy     0         177              0.000000        2   
1  A Ashish Reddy    10         100              0.100000        1   
2  A Ashish Reddy     0         186              0.000000        1   
3  A Ashish Reddy     3         150              0.020000        1   
4  A Ashish Reddy     0         181              0.000000        1   
5  A Ashish Reddy     8         145              0.055172        2   
6  A Ashish Reddy     0         187              0.000000        0   
7  A Ashish Reddy    10         128              0.078125        0   
8  A Ashish Reddy     4         132              0.030303        3   
9  A Ashish Reddy     7         126              0.055556        1   

   total_wickets  bowling_contribution  
0              7              0.285714  
1              5              0.200000  
2              5              0.200000  
3              6              0.166667  
4              5        

In [1087]:
print(df[df["player"] == "BB McCullum"][["player", "runs", "team_total", "batting_contribution"]].head(3))

           player  runs  team_total  batting_contribution
3623  BB McCullum   158         222              0.711712
3624  BB McCullum     5         112              0.044643
3625  BB McCullum    24         147              0.163265


In [1088]:
df["rolling_batting_contribution_5"] = (
    df.groupby("player")["batting_contribution"].transform(lambda x: x.shift(1).rolling(5, min_periods = 1).mean())
)
df["rolling_bowling_contribution_5"] = (
    df.groupby("player")["bowling_contribution"].transform(lambda x: x.shift(1).rolling(5,min_periods = 1).mean())
)


In [1089]:
print(df[df["player"] == "BB McCullum"][["player", "runs", "team_total", "rolling_batting_contribution_5"]].head(10))

           player  runs  team_total  rolling_batting_contribution_5
3623  BB McCullum   158         222                             NaN
3624  BB McCullum     5         112                        0.711712
3625  BB McCullum    24         147                        0.378177
3626  BB McCullum     1         137                        0.306540
3627  BB McCullum     1         101                        0.231730
3628  BB McCullum    21          79                        0.187364
3629  BB McCullum     4         183                        0.098186
3630  BB McCullum     1          95                        0.093629
3631  BB McCullum     0         139                        0.063081
3632  BB McCullum     5         139                        0.061622


In [1090]:
df[df["player"] == "V Kohli"][["player", "date", "total_fantasy_points", "rolling_avg_fantasy_3", 
    "rolling_avg_fantasy_5", "rolling_avg_fantasy_10", 
    "rolling_avg_runs_5", "rolling_avg_wickets_5", 
    "matches_played","rolling_std_fantasy_5","rolling_std_fantasy_10","expanding_season_fantasy_avg","expanding_season_fantasy_std"]].head(30)

,player,date,total_fantasy_points,rolling_avg_fantasy_3,rolling_avg_fantasy_5,rolling_avg_fantasy_10,rolling_avg_runs_5,rolling_avg_wickets_5,matches_played,rolling_std_fantasy_5,rolling_std_fantasy_10,expanding_season_fantasy_avg,expanding_season_fantasy_std
25844,V Kohli,2008-04-18,5,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN
25845,V Kohli,2008-04-20,36,5.000000,5.0,5.000000,1.000000,0.0,1.0,NaN,NaN,5.000000,NaN
25846,V Kohli,2008-04-26,19,20.500000,20.5,20.500000,12.000000,0.0,2.0,21.920310,21.920310,20.500000,21.920310
25847,V Kohli,2008-04-28,18,20.000000,20.0,20.000000,12.333333,0.0,3.0,15.524175,15.524175,20.000000,15.524175
25848,V Kohli,2008-04-30,-1,24.333333,19.5,19.500000,12.250000,0.0,4.0,12.714821,12.714821,19.500000,12.714821
25849,V Kohli,2008-05-03,49,12.000000,15.4,15.400000,10.000000,0.0,5.0,14.328294,14.328294,15.400000,14.328294
25850,V Kohli,2008-05-05,43,22.000000,24.2,21.000000,17.400000,0.0,6.0,19.070920,19.070920,21.000000,18.772320
25851,V Kohli,2008-05-08,4,30.333333,25.6,24.142857,19.600000,0.0,7.0,20.366639,20.366639,24.142857,19.047560
25852,V Kohli,2008-05-12,23,32.000000,22.6,21.625000,17.000000,0.0,8.0,22.567676,22.567676,21.625000,19.018318
25853,V Kohli,2008-05-17,7,23.333333,23.6,21.777778,18.800000,0.0,9.0,22.423202,22.423202,21.777778,17.795911


In [1091]:
HOME_CITIES = { 
    "Mumbai Indians": "Mumbai",
    "Chennai Super Kings": "Chennai",
    "Royal Challengers Bangalore": "Bengaluru",
    "Royal Challengers Bengaluru": "Bengaluru",
    "Kolkata Knight Riders": "Kolkata",
    "Sunrisers Hyderabad": "Hyderabad",
    "Rajasthan Royals": "Jaipur",
    "Punjab Kings": "Mohali",
    "Kings XI Punjab": "Mohali",
    "Delhi Capitals": "Delhi",
    "Delhi Daredevils": "Delhi",
    "Gujarat Titans": "Ahmedabad",
    "Lucknow Super Giants": "Lucknow",
    "Deccan Chargers": "Hyderabad",
    "Kochi Tuskers Kerala": "Kochi",
    "Pune Warriors": "Pune",
    "Rising Pune Supergiants": "Pune"
}

In [1092]:
df["is_home"] = (df["team"].map(HOME_CITIES) == df["city"]).astype(int)

In [1093]:
df[["team", "city", "is_home"]]


,team,city,is_home
0,Deccan Chargers,Pune,0
1,Deccan Chargers,Mumbai,0
2,Deccan Chargers,Cuttack,0
3,Deccan Chargers,Chennai,0
4,Deccan Chargers,Bangalore,0
...,...,...,...
27904,Sunrisers Hyderabad,Hyderabad,1
27905,Sunrisers Hyderabad,Chennai,0
27906,Sunrisers Hyderabad,Ahmedabad,0
27907,Sunrisers Hyderabad,Hyderabad,1


In [1094]:
def venue_avg_before(group):
    return group.shift(1).expanding().mean()

def venue_std_before(group):
    return group.shift(1).expanding().std()


df["venue_avg_fantasy"] = (
    df.groupby(["player","venue"])["total_fantasy_points"]
    .transform(venue_avg_before)
    )

df["venue_std_fantasy"] = (
    df.groupby(["player","venue"])["total_fantasy_points"]
    .transform(venue_std_before)
)



df[["player", "date", "venue", "total_fantasy_points", "venue_avg_fantasy", "venue_std_fantasy"]].head(30)

,player,date,venue,total_fantasy_points,venue_avg_fantasy,venue_std_fantasy
0,A Ashish Reddy,2012-04-26,Subrata Roy Sahara Stadium,28,NaN,NaN
1,A Ashish Reddy,2012-04-29,Wankhede Stadium,32,NaN,NaN
2,A Ashish Reddy,2012-05-01,Barabati Stadium,10,NaN,NaN
3,A Ashish Reddy,2012-05-04,"MA Chidambaram Stadium, Chepauk",19,NaN,NaN
4,A Ashish Reddy,2012-05-06,M Chinnaswamy Stadium,10,NaN,NaN
5,A Ashish Reddy,2012-05-08,"Rajiv Gandhi International Stadium, Uppal",31,NaN,NaN
6,A Ashish Reddy,2012-05-10,"Rajiv Gandhi International Stadium, Uppal",-8,31.000000,NaN
7,A Ashish Reddy,2012-05-18,"Rajiv Gandhi International Stadium, Uppal",16,11.500000,27.577164
8,A Ashish Reddy,2012-05-20,"Rajiv Gandhi International Stadium, Uppal",46,13.000000,19.672316
9,A Ashish Reddy,2013-04-05,"Rajiv Gandhi International Stadium, Uppal",18,21.250000,23.027158


In [1095]:
career_stats = df.groupby("player").agg(
    total_matches = ("player", "count"),
    career_runs = ("runs", "sum"),
    career_balls_bowled = ("balls_bowled","sum"),
    career_wickets = ("wickets", "sum"),
    career_balls_faced = ("balls_faced", "sum"),
    career_runs_conceded = ("runs_conceded","sum"),
    avg_bat_position = ("batting_position", lambda x: x.mode()[0])
).reset_index()

print(career_stats[career_stats["player"].isin(["V Kohli", "JJ Bumrah", "HH Pandya"])])

        player  total_matches  career_runs  career_balls_bowled  \
250  HH Pandya            162         2964                 1764   
295  JJ Bumrah            158           75                 3653   
761    V Kohli            282         9346                  251   

     career_wickets  career_balls_faced  career_runs_conceded  \
250              82                2025                  2751   
295             190                  86                  4469   
761               4                6930                   368   

     avg_bat_position  
250                 6  
295                 0  
761                 3  


In [1096]:
career_stats["career_strike_rate"] = (
    (career_stats["career_runs"] / career_stats["career_balls_faced"]) * 100 
).fillna(0)

career_stats["career_economy"] = (
    career_stats["career_runs_conceded"] / career_stats["career_balls_bowled"] * 6
).fillna(0)

career_stats["career_batting_avg"] = (
    career_stats["career_runs"] / career_stats["total_matches"]
).fillna(0)

print(career_stats[career_stats["player"].isin(["V Kohli", "JJ Bumrah", "HH Pandya"])])

        player  total_matches  career_runs  career_balls_bowled  \
250  HH Pandya            162         2964                 1764   
295  JJ Bumrah            158           75                 3653   
761    V Kohli            282         9346                  251   

     career_wickets  career_balls_faced  career_runs_conceded  \
250              82                2025                  2751   
295             190                  86                  4469   
761               4                6930                   368   

     avg_bat_position  career_strike_rate  career_economy  career_batting_avg  
250                 6          146.370370        9.357143           18.296296  
295                 0           87.209302        7.340268            0.474684  
761                 3          134.862915        8.796813           33.141844  


In [1097]:
df = df.merge(
    career_stats[["player", "career_strike_rate", "career_economy", "career_batting_avg"]],
    on="player", how="left"
)

df[["career_strike_rate", "career_economy", "career_batting_avg"]] = df[["career_strike_rate", "career_economy", "career_batting_avg"]].fillna(0)

In [1098]:
career_stats.to_csv("data/processed/career_stats.csv", index=False)
print("Saved Career Stats")

Saved Career Stats


In [1099]:
def classify_role(total_matches, career_runs, career_wickets, avg_bat_position,career_balls_bowled):
    is_genuine_bowler = career_wickets >= 20 and career_balls_bowled >= 200
    is_genuine_batter = is_genuine_batter = career_runs >= 500 or (career_runs >= 300 and avg_bat_position <= 7)

    if total_matches > 20:
        if is_genuine_batter and is_genuine_bowler:
            role = "allrounder"
        elif is_genuine_bowler:
            role = "bowler"
        elif is_genuine_batter:
            role = "batter"
        else:
            role = "unknown"
    else:
        if avg_bat_position == 0:
            role = "bowler"
        elif career_balls_bowled > 50 and avg_bat_position >= 7:
            role = "bowler"
        elif avg_bat_position <= 7 and career_balls_bowled > 30:
            role = "allrounder"
        elif avg_bat_position <= 7:
            role = "batter"
        else:
            role = "bowler"
        
    return role

In [1100]:
career_stats["role"] = career_stats.apply(
    lambda row: classify_role(
        row["total_matches"],
        row["career_runs"],
        row["career_wickets"],
        row["avg_bat_position"],
        row["career_balls_bowled"]
    ),
    axis=1
)

In [1101]:

print(career_stats[career_stats["player"].isin(["V Kohli", "JJ Bumrah", "HH Pandya"])])

        player  total_matches  career_runs  career_balls_bowled  \
250  HH Pandya            162         2964                 1764   
295  JJ Bumrah            158           75                 3653   
761    V Kohli            282         9346                  251   

     career_wickets  career_balls_faced  career_runs_conceded  \
250              82                2025                  2751   
295             190                  86                  4469   
761               4                6930                   368   

     avg_bat_position  career_strike_rate  career_economy  career_batting_avg  \
250                 6          146.370370        9.357143           18.296296   
295                 0           87.209302        7.340268            0.474684   
761                 3          134.862915        8.796813           33.141844   

           role  
250  allrounder  
295      bowler  
761      batter  


In [1102]:
print(career_stats["role"].value_counts())

role
bowler        414
batter        294
allrounder     74
unknown        29
Name: count, dtype: int64


In [1103]:
df = df.drop(columns=["role"], errors="ignore")
df = df.merge(career_stats[["player", "role"]], on="player", how="left")

In [1104]:
print(df[["player", "role"]].head(10))
print(df["role"].value_counts())

           player     role
0  A Ashish Reddy  unknown
1  A Ashish Reddy  unknown
2  A Ashish Reddy  unknown
3  A Ashish Reddy  unknown
4  A Ashish Reddy  unknown
5  A Ashish Reddy  unknown
6  A Ashish Reddy  unknown
7  A Ashish Reddy  unknown
8  A Ashish Reddy  unknown
9  A Ashish Reddy  unknown
role
batter        12420
bowler         8682
allrounder     6060
unknown         747
Name: count, dtype: int64


In [1105]:
df["role_encoded"] = df["role"].map({
    "batter": 0,
    "bowler": 1,
    "allrounder": 2,
    "unknown": 3
})

In [1106]:
def oppositon_avg_before(group):
    return group.shift(1).expanding().mean()

def oppositon_std_before(group):
    return group.shift(1).expanding().std()

df["opposition_avg_fantasy"] = (
    df.groupby(["player", "opposition"])["total_fantasy_points"]
    .transform(oppositon_avg_before)
    )

df["opposition_std_fantasy"] = (
    df.groupby(["player", "opposition"])["total_fantasy_points"]
    .transform(oppositon_std_before)
    )

print("Opposition feature added")
print(df.shape)
print(df.isnull().sum())

Opposition feature added
(27909, 52)
match_id                              0
player                                0
team                                  0
opposition                            0
venue                                 0
city                                  0
date                                  0
season                                0
toss_winner                           0
toss_decision                         0
runs                                  0
balls_faced                           0
fours                                 0
sixes                                 0
strike_rate                           0
batting_position                      0
wickets                               0
runs_conceded                         0
balls_bowled                          0
maidens                               0
economy                               0
total_wickets                         0
player_innings                        0
team_total                            0
tot

In [1107]:
match_innings = df[df["player_innings"] != 0].drop_duplicates(subset = ["match_id","player_innings"])[
       ["match_id", "venue", "date", "player_innings", "team_total"]  
    ]

In [1108]:
match_pivot = match_innings.pivot(index = "match_id", columns= "player_innings",values= "team_total")
match_pivot.columns = ["innings1_score", "innings2_score"]
match_pivot = match_pivot.reset_index()

print(match_pivot.head(10))

   match_id  innings1_score  innings2_score
0    335982           222.0            82.0
1    335983           240.0           207.0
2    335984           129.0           132.0
3    335985           165.0           166.0
4    335986           110.0           112.0
5    335987           166.0           168.0
6    335988           142.0           143.0
7    335989           208.0           202.0
8    335990           214.0           217.0
9    335991           182.0           116.0


In [1109]:
match_meta = df.drop_duplicates(subset="match_id")[["match_id","venue","date"]]
match_level = match_pivot.merge(match_meta, on = "match_id", how = "left")
match_level["date"] = pd.to_datetime(match_level["date"])
match_level = match_level.sort_values("date").reset_index(drop = True)

print(match_level.head(10))
print(match_level.shape)

   match_id  innings1_score  innings2_score  \
0    335982           222.0            82.0   
1    335983           240.0           207.0   
2    335984           129.0           132.0   
3    335985           165.0           166.0   
4    335986           110.0           112.0   
5    335987           166.0           168.0   
6    335988           142.0           143.0   
7    335989           208.0           202.0   
8    335990           214.0           217.0   
9    335991           182.0           116.0   

                                        venue       date  
0                       M Chinnaswamy Stadium 2008-04-18  
1  Punjab Cricket Association Stadium, Mohali 2008-04-19  
2                            Feroz Shah Kotla 2008-04-19  
3                            Wankhede Stadium 2008-04-20  
4                                Eden Gardens 2008-04-20  
5                      Sawai Mansingh Stadium 2008-04-21  
6   Rajiv Gandhi International Stadium, Uppal 2008-04-22  
7         

In [1110]:
match_level["venue_avg_innings1"] = (
    match_level.groupby("venue")["innings1_score"]
    .transform(lambda x : x.shift(1).expanding().mean())
)

match_level["venue_avg_innings2"] = (
    match_level.groupby("venue")["innings2_score"]
    .transform(lambda x : x.shift(1).expanding().mean())
)

match_level["venue_avg_total_runs"] = (
    match_level.groupby("venue").apply(
        lambda g : (g["innings1_score"] + g["innings2_score"]).shift(1).expanding().mean()
    ).reset_index(level = 0, drop = True)
)

print(match_level[["venue", "date", "innings1_score", "innings2_score", "venue_avg_innings1", "venue_avg_innings2"]].head(20))

                                         venue       date  innings1_score  \
0                        M Chinnaswamy Stadium 2008-04-18           222.0   
1   Punjab Cricket Association Stadium, Mohali 2008-04-19           240.0   
2                             Feroz Shah Kotla 2008-04-19           129.0   
3                             Wankhede Stadium 2008-04-20           165.0   
4                                 Eden Gardens 2008-04-20           110.0   
5                       Sawai Mansingh Stadium 2008-04-21           166.0   
6    Rajiv Gandhi International Stadium, Uppal 2008-04-22           142.0   
7              MA Chidambaram Stadium, Chepauk 2008-04-23           208.0   
8    Rajiv Gandhi International Stadium, Uppal 2008-04-24           214.0   
9   Punjab Cricket Association Stadium, Mohali 2008-04-25           182.0   
10                       M Chinnaswamy Stadium 2008-04-26           135.0   
11             MA Chidambaram Stadium, Chepauk 2008-04-26           147.0   

In [1111]:
venue_features = match_level[["match_id", "venue_avg_innings1", "venue_avg_innings2", "venue_avg_total_runs"]]
df = df.merge(venue_features, on = "match_id", how = "left")

print(df.shape)
print(df[["player", "match_id", "venue", "venue_avg_innings1", "venue_avg_innings2"]].head(10))

(27909, 55)
           player  match_id                                      venue  \
0  A Ashish Reddy    548341                 Subrata Roy Sahara Stadium   
1  A Ashish Reddy    548346                           Wankhede Stadium   
2  A Ashish Reddy    548348                           Barabati Stadium   
3  A Ashish Reddy    548352            MA Chidambaram Stadium, Chepauk   
4  A Ashish Reddy    548356                      M Chinnaswamy Stadium   
5  A Ashish Reddy    548359  Rajiv Gandhi International Stadium, Uppal   
6  A Ashish Reddy    548329  Rajiv Gandhi International Stadium, Uppal   
7  A Ashish Reddy    548373  Rajiv Gandhi International Stadium, Uppal   
8  A Ashish Reddy    548376  Rajiv Gandhi International Stadium, Uppal   
9  A Ashish Reddy    598000  Rajiv Gandhi International Stadium, Uppal   

   venue_avg_innings1  venue_avg_innings2  
0          155.666667          149.333333  
1          150.157895          139.000000  
2          155.666667          150.666667

In [1112]:
unique_cities = df["city"].dropna().unique()
print(len(unique_cities))
print(unique_cities)

38
<StringArray>
[          'Pune',         'Mumbai',        'Cuttack',        'Chennai',
      'Bangalore',      'Hyderabad',          'Delhi',        'Kolkata',
         'Jaipur',  'Visakhapatnam',     'Chandigarh',        'Lucknow',
      'Bengaluru',      'Ahmedabad',     'Dharamsala', 'New Chandigarh',
      'Cape Town',         'Durban', 'Port Elizabeth',       'Guwahati',
   'Johannesburg',      'Centurion',         'Nagpur',    'Navi Mumbai',
    'East London',   'Bloemfontein',          'Kochi',      'Abu Dhabi',
        'Unknown',         'Raipur',         'Rajkot',         'Kanpur',
         'Ranchi',          'Dubai',        'Sharjah',         'Mohali',
      'Kimberley',         'Indore']
Length: 38, dtype: str


In [1113]:
import requests

CITY_NAME_OVERRIDES = {
    "Bangalore": "Bengaluru",
    "New Chandigarh": "Mullanpur",
}


def geocode_city(city_name, country_code = None):
    city_name = CITY_NAME_OVERRIDES.get(city_name,city_name)
    url = "https://geocoding-api.open-meteo.com/v1/search"
    params = {"name": city_name, "count": 5}
    response = requests.get(url, params = params, timeout= 10)
    if response.status_code == 200:
        data = response.json()
        if "results" in data:
            for result in data["results"]:
                if country_code is None or result.get("country_code") == country_code:
                    return result["latitude"], result["longitude"]
    return None , None
    

In [1114]:
print(geocode_city("Bangalore", country_code="IN"))
print(geocode_city("New Chandigarh", country_code="IN"))
print(geocode_city("Mumbai", country_code="IN"))

(12.97194, 77.59369)
(30.85457, 75.66089)
(19.07283, 72.88261)


In [1115]:
import time

INDIAN_CITIES = {
    "Bangalore", "Delhi", "Chandigarh", "Mumbai", "Kolkata", "Jaipur", "Hyderabad",
    "Chennai", "Ahmedabad", "Cuttack", "Nagpur", "Dharamsala", "Kochi", "Indore",
    "Visakhapatnam", "Pune", "Raipur", "Ranchi", "Rajkot", "Kanpur", "Bengaluru",
    "Navi Mumbai", "Lucknow", "Guwahati", "Mohali", "New Chandigarh"
}

city_coords = {}

for city in unique_cities:
    if city == "Unknown":
        continue
    country_code = "IN" if city in INDIAN_CITIES else None
    lat , lon = geocode_city(city,country_code = country_code)
    city_coords[city] = {"lat": lat , "lon": lon}
    print(city, lat, lon)
    time.sleep(0.2)

Pune 18.51957 73.85535
Mumbai 19.07283 72.88261
Cuttack 20.46497 85.87927
Chennai 13.08784 80.27847
Bangalore 12.97194 77.59369
Hyderabad 17.38405 78.45636
Delhi 28.65195 77.23149
Kolkata 22.56263 88.36304
Jaipur 26.91962 75.78781
Visakhapatnam 17.68009 83.20161
Chandigarh 30.73629 76.7884
Lucknow 26.83928 80.92313
Bengaluru 12.97194 77.59369
Ahmedabad 23.02579 72.58727
Dharamsala 32.22006 76.32013
New Chandigarh 30.85457 75.66089
Cape Town -33.92584 18.42322
Durban -29.8579 31.0292
Port Elizabeth -33.96109 25.61494
Guwahati 26.1844 91.7458
Johannesburg -26.20227 28.04363
Centurion -25.85891 28.18577
Nagpur 21.14631 79.08491
Navi Mumbai 19.03681 73.01582
East London -33.01529 27.91162
Bloemfontein -29.12107 26.214
Kochi 9.93988 76.26022
Abu Dhabi 24.45118 54.39696
Raipur 21.23333 81.63333
Rajkot 22.29161 70.79322
Kanpur 26.46523 80.34975
Ranchi 23.34316 85.3094
Dubai 25.07725 55.30927
Sharjah 25.3342 55.41221
Mohali 30.67995 76.72211
Kimberley -28.73226 24.76232
Indore 22.71792 75.8333

In [1116]:
city_date_ranges = df.groupby("city")["date"].agg(["min","max"])
print(city_date_ranges)

                      min        max
city                                
Abu Dhabi      2014-04-16 2021-10-08
Ahmedabad      2010-03-15 2026-05-31
Bangalore      2008-04-18 2017-05-19
Bengaluru      2017-04-08 2026-04-24
Bloemfontein   2009-05-15 2009-05-17
Cape Town      2009-04-18 2009-04-26
Centurion      2009-04-28 2009-05-22
Chandigarh     2008-04-19 2023-05-03
Chennai        2008-04-23 2026-05-18
Cuttack        2010-03-19 2014-05-14
Delhi          2008-04-19 2026-05-17
Dharamsala     2010-04-16 2026-05-26
Dubai          2021-09-19 2021-10-15
Durban         2009-04-21 2009-05-20
East London    2009-05-01 2009-05-08
Guwahati       2023-04-05 2026-04-10
Hyderabad      2008-04-22 2026-05-22
Indore         2011-05-13 2018-05-14
Jaipur         2008-04-21 2026-05-19
Johannesburg   2009-05-02 2009-05-24
Kanpur         2016-05-19 2017-05-13
Kimberley      2009-05-09 2009-05-11
Kochi          2011-04-09 2011-05-05
Kolkata        2008-04-20 2026-05-24
Lucknow        2023-04-01 2026-05-23
M

In [1117]:
def fetch_city_weather_range(lat,lon,start_date,end_date):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date":start_date,
        "end_date": end_date,
        "hourly": "temperature_2m,relative_humidity_2m,dew_point_2m,wind_speed_10m,precipitation",
        "timezone": "auto"
    }
    response = requests.get(url,params = params, timeout = 30)
    if response.status_code == 200:
        return response.json()
    else:
        print("Failed:", response.status_code, response.text[:200])
        return None

In [1118]:
import pandas as pd

def extract_match_day_weather(weather_json, match_date):
    times = weather_json["hourly"]["time"]
    temps = weather_json["hourly"]["temperature_2m"]
    humidity = weather_json["hourly"]["relative_humidity_2m"]
    dew = weather_json["hourly"]["dew_point_2m"]
    wind = weather_json["hourly"]["wind_speed_10m"]
    precip = weather_json["hourly"]["precipitation"]

    match_temps, match_humidity, match_dew, match_wind, match_precip = [],[],[],[],[]

    for i , t in enumerate(times):
        if t.startswith(match_date):
            hour = int(t.split("T")[1].split(":")[0])
            if 15 <= hour <= 21:
                match_temps.append(temps[i])
                match_humidity.append(humidity[i])
                match_dew.append(dew[i])
                match_wind.append(wind[i])
                match_precip.append(precip[i])
    if not  match_temps:
        return None
 
    return {
        "temp": sum(match_temps) / len(match_temps),
        "humidity": sum(match_humidity)/ len(match_humidity),
        "dew": sum(match_dew) / len(match_dew),
        "windspeed": sum(match_wind) / len(match_wind),
        "precip": sum(match_precip)
    }


In [1119]:
import json
import time as time_module

# reload city_coords, which we did NOT lose
with open("data/processed/city_coords.json") as f:
    city_coords = json.load(f)

weather_lookup = {}
fetched_cities = set()

for city in unique_cities:
    if city == "Unknown" or city in fetched_cities:
        continue

    coords = city_coords.get(city)
    if coords is None or coords["lat"] is None:
        print(f"Skipping {city}, no coordinates")
        continue

    city_matches = df[df["city"] == city]
    start_date = city_matches["date"].min().strftime("%Y-%m-%d")
    end_date = city_matches["date"].max().strftime("%Y-%m-%d")

    print(f"Fetching {city}: {start_date} to {end_date}")
    weather_json = fetch_city_weather_range(coords["lat"], coords["lon"], start_date, end_date)

    if weather_json is None:
        print(f"Failed to fetch {city}, retrying in 20s")
        time_module.sleep(20)
        weather_json = fetch_city_weather_range(coords["lat"], coords["lon"], start_date, end_date)
        if weather_json is None:
            print(f"Skipping {city} after retry failure")
            continue

    unique_dates_for_city = city_matches["date"].dt.strftime("%Y-%m-%d").unique()
    for match_date in unique_dates_for_city:
        day_weather = extract_match_day_weather(weather_json, match_date)
        weather_lookup[f"{city},{match_date}"] = day_weather

    fetched_cities.add(city)

    # save progress after EVERY city, to a fixed filename that's safe to overwrite mid-run
    with open("data/processed/weather_lookup_progress.json", "w") as f:
        json.dump(weather_lookup, f)

    time_module.sleep(3)

print(f"Done. Total weather entries: {len(weather_lookup)}, cities fetched: {len(fetched_cities)}")

# final save to a clearly versioned, permanent filename
with open("data/processed/weather_lookup_final.json", "w") as f:
    json.dump(weather_lookup, f)
print("Saved final weather lookup")

Fetching Pune: 2012-04-08 to 2022-05-14
Fetching Mumbai: 2008-04-20 to 2026-05-24
Fetching Cuttack: 2010-03-19 to 2014-05-14
Fetching Chennai: 2008-04-23 to 2026-05-18
Fetching Bangalore: 2008-04-18 to 2017-05-19
Fetching Hyderabad: 2008-04-22 to 2026-05-22
Fetching Delhi: 2008-04-19 to 2026-05-17
Fetching Kolkata: 2008-04-20 to 2026-05-24
Fetching Jaipur: 2008-04-21 to 2026-05-19
Fetching Visakhapatnam: 2012-04-07 to 2025-03-30
Failed: 429 {"reason":"Minutely API request limit exceeded. Please try again in one minute.","error":true}
Failed to fetch Visakhapatnam, retrying in 20s
Fetching Chandigarh: 2008-04-19 to 2023-05-03
Fetching Lucknow: 2023-04-01 to 2026-05-23
Fetching Bengaluru: 2017-04-08 to 2026-04-24
Fetching Ahmedabad: 2010-03-15 to 2026-05-31
Fetching Dharamsala: 2010-04-16 to 2026-05-26
Fetching New Chandigarh: 2025-05-29 to 2026-05-29
Fetching Cape Town: 2009-04-18 to 2009-04-26
Fetching Durban: 2009-04-21 to 2009-05-20
Fetching Port Elizabeth: 2009-04-20 to 2009-05-16
F

In [1120]:
import json
with open("data/processed/city_coords.json", "w") as f:
    json.dump(city_coords, f)
print("Saved city coordinates for future live lookups")

Saved city coordinates for future live lookups


In [1121]:
df["weather_key"] = df["city"] + "," + df["date"].dt.strftime("%Y-%m-%d")

df["weather_temp"] = df["weather_key"].map(lambda k: weather_lookup.get(k,{}).get("temp") if weather_lookup.get(k) else None)
df["weather_humidity"] = df["weather_key"].map(lambda k: weather_lookup.get(k,{}).get("humidity") if weather_lookup.get(k) else None)
df["weather_dew"] = df["weather_key"].map(lambda k: weather_lookup.get(k,{}).get("dew") if weather_lookup.get(k) else None)
df["weather_windspeed"] = df["weather_key"].map(lambda k: weather_lookup.get(k,{}).get("windspeed") if weather_lookup.get(k) else None)
df["weather_precip"] = df["weather_key"].map(lambda k: weather_lookup.get(k,{}).get("precip") if weather_lookup.get(k) else None)

df = df.drop(columns = ["weather_key"])
print(df[["weather_temp","weather_humidity","weather_dew","weather_windspeed","weather_precip"]])

       weather_temp  weather_humidity  weather_dew  weather_windspeed  \
0         34.014286         25.857143    10.442857          15.557143   
1         28.028571         70.142857    21.900000          14.671429   
2         33.428571         61.571429    24.514286          12.671429   
3         32.500000         65.285714    24.614286          23.142857   
4         28.328571         38.142857    12.342857           9.985714   
...             ...               ...          ...                ...   
27904     36.000000         26.714286    13.714286           7.928571   
27905     31.200000         71.428571    25.257143          16.142857   
27906     40.542857         13.285714     6.285714          11.585714   
27907     35.614286         20.571429     9.442857           7.485714   
27908     35.528571         42.571429    20.557143          11.071429   

       weather_precip  
0                 0.0  
1                 0.0  
2                 0.1  
3                 0.0  
4  

In [1122]:
print(df[["weather_temp", "weather_humidity", "weather_dew", "weather_windspeed", "weather_precip"]].isnull().sum())

weather_temp         1122
weather_humidity     1122
weather_dew          1122
weather_windspeed    1122
weather_precip       1122
dtype: int64


In [1123]:
missing_weather = df[df["weather_temp"].isnull()]
print(missing_weather["city"].value_counts())


city
Unknown    1122
Name: count, dtype: int64


In [1124]:
for col in ["weather_temp", "weather_humidity", "weather_dew", "weather_windspeed", "weather_precip"]:
    df[col] = df[col].fillna(df[col].mean())

print(df[["weather_temp", "weather_humidity", "weather_dew", "weather_windspeed", "weather_precip"]].isnull().sum())

weather_temp         0
weather_humidity     0
weather_dew          0
weather_windspeed    0
weather_precip       0
dtype: int64


In [1125]:
df.to_csv("data/processed/player_match_features.csv", index=False)
print("Saved")

Saved


In [1126]:
df["venue_avg_innings1"] = df["venue_avg_innings1"].fillna(df["venue_avg_innings1"].mean())
df["venue_avg_innings2"] = df["venue_avg_innings2"].fillna(df["venue_avg_innings2"].mean())
df["venue_avg_total_runs"] = df["venue_avg_total_runs"].fillna(df["venue_avg_total_runs"].mean())

In [1127]:
df["venue_first_appearance"] = df["venue_avg_fantasy"].isnull().astype(int)
df["opposition_first_appearance"] = df["opposition_avg_fantasy"].isnull().astype(int)

df["venue_avg_fantasy"] = df["venue_avg_fantasy"].fillna(0)
df["venue_std_fantasy"] = df["venue_std_fantasy"].fillna(0)

df["opposition_avg_fantasy"] = df["opposition_avg_fantasy"].fillna(0)
df["opposition_std_fantasy"] = df["opposition_std_fantasy"].fillna(0)

df["expanding_season_fantasy_avg"] = df["expanding_season_fantasy_avg"].fillna(0)
df["expanding_season_fantasy_std"] = df["expanding_season_fantasy_std"].fillna(0)

df["rolling_batting_contribution_5"] = df["rolling_batting_contribution_5"].fillna(0)
df["rolling_bowling_contribution_5"] = df["rolling_bowling_contribution_5"].fillna(0)

df["rolling_avg_fantasy_3"] = df["rolling_avg_fantasy_3"].fillna(0)
df["rolling_avg_fantasy_5"] = df["rolling_avg_fantasy_5"].fillna(0)
df["rolling_avg_fantasy_10"] = df["rolling_avg_fantasy_10"].fillna(0)
df["rolling_std_fantasy_5"] = df["rolling_std_fantasy_5"].fillna(0)
df["rolling_std_fantasy_10"] = df["rolling_std_fantasy_10"].fillna(0)
df["rolling_avg_runs_5"] = df["rolling_avg_runs_5"].fillna(0)
df["rolling_avg_wickets_5"] = df["rolling_avg_wickets_5"].fillna(0)
df["Total_career_runs"] = df["Total_career_runs"].fillna(0)
df["Total_career_wickets"] = df["Total_career_wickets"].fillna(0)


print(df.isnull().sum())

match_id                       0
player                         0
team                           0
opposition                     0
venue                          0
                              ..
weather_dew                    0
weather_windspeed              0
weather_precip                 0
venue_first_appearance         0
opposition_first_appearance    0
Length: 62, dtype: int64


In [1128]:
import os
os.makedirs("data/processed", exist_ok= True)
df.to_csv("data/processed/player_match_features.csv", index = False)
print(len(df))
df["won_toss"] = (df["team"] == df["toss_winner"]).astype(int)
print(df["won_toss"])

27909
0        1
1        0
2        1
3        0
4        0
        ..
27904    0
27905    1
27906    1
27907    1
27908    1
Name: won_toss, Length: 27909, dtype: int64


In [1129]:
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date")

train = df[df["date"].dt.year < 2025]
test = df[df["date"].dt.year >=2025]

print(train.shape)
print(test.shape)
print(f"Train: {len(train)/len(df)*100:.1f}%")
print(f"Test: {len(test)/len(df)*100:.1f}%")

(24367, 63)
(3542, 63)
Train: 87.3%
Test: 12.7%


In [1130]:
print(len(train))
print(len(test))

24367
3542


In [1131]:
print(train.columns)

Index(['match_id', 'player', 'team', 'opposition', 'venue', 'city', 'date',
       'season', 'toss_winner', 'toss_decision', 'runs', 'balls_faced',
       'fours', 'sixes', 'strike_rate', 'batting_position', 'wickets',
       'runs_conceded', 'balls_bowled', 'maidens', 'economy', 'total_wickets',
       'player_innings', 'team_total', 'total_fantasy_points',
       'batting_position_bucket', 'Total_career_runs', 'Total_career_wickets',
       'rolling_avg_fantasy_5', 'rolling_std_fantasy_5',
       'rolling_avg_fantasy_3', 'rolling_std_fantasy_10',
       'rolling_avg_fantasy_10', 'rolling_avg_runs_5', 'rolling_avg_wickets_5',
       'matches_played', 'expanding_season_fantasy_avg',
       'expanding_season_fantasy_std', 'batting_contribution',
       'bowling_contribution', 'rolling_batting_contribution_5',
       'rolling_bowling_contribution_5', 'is_home', 'venue_avg_fantasy',
       'venue_std_fantasy', 'career_strike_rate', 'career_economy',
       'career_batting_avg', 'role', 'r

In [1132]:
df[["team", "toss_winner", "won_toss"]].head(20)

,team,toss_winner,won_toss
12415,Kolkata Knight Riders,Royal Challengers Bangalore,0
3623,Kolkata Knight Riders,Royal Challengers Bangalore,0
8082,Kolkata Knight Riders,Royal Challengers Bangalore,0
16408,Royal Challengers Bangalore,Royal Challengers Bangalore,1
18109,Royal Challengers Bangalore,Royal Challengers Bangalore,1
12112,Kolkata Knight Riders,Royal Challengers Bangalore,0
15092,Kolkata Knight Riders,Royal Challengers Bangalore,0
9254,Royal Challengers Bangalore,Royal Challengers Bangalore,1
720,Kolkata Knight Riders,Royal Challengers Bangalore,0
719,Royal Challengers Bangalore,Royal Challengers Bangalore,1


In [1133]:
df.columns

Index(['match_id', 'player', 'team', 'opposition', 'venue', 'city', 'date',
       'season', 'toss_winner', 'toss_decision', 'runs', 'balls_faced',
       'fours', 'sixes', 'strike_rate', 'batting_position', 'wickets',
       'runs_conceded', 'balls_bowled', 'maidens', 'economy', 'total_wickets',
       'player_innings', 'team_total', 'total_fantasy_points',
       'batting_position_bucket', 'Total_career_runs', 'Total_career_wickets',
       'rolling_avg_fantasy_5', 'rolling_std_fantasy_5',
       'rolling_avg_fantasy_3', 'rolling_std_fantasy_10',
       'rolling_avg_fantasy_10', 'rolling_avg_runs_5', 'rolling_avg_wickets_5',
       'matches_played', 'expanding_season_fantasy_avg',
       'expanding_season_fantasy_std', 'batting_contribution',
       'bowling_contribution', 'rolling_batting_contribution_5',
       'rolling_bowling_contribution_5', 'is_home', 'venue_avg_fantasy',
       'venue_std_fantasy', 'career_strike_rate', 'career_economy',
       'career_batting_avg', 'role', 'r

In [1134]:
df.shape

(27909, 63)

In [1135]:
features = [
"rolling_avg_fantasy_5",
"rolling_avg_fantasy_10",
"rolling_std_fantasy_10",
"rolling_avg_runs_5",
"rolling_avg_wickets_5",
"batting_position",
"matches_played",
"venue_std_fantasy",
"opposition_std_fantasy",
"venue_first_appearance",
"opposition_first_appearance",
"career_strike_rate",
"won_toss",
"expanding_season_fantasy_std",
"is_home",
"role_encoded",
"rolling_bowling_contribution_5",
"rolling_batting_contribution_5",
"venue_avg_innings1",
"venue_avg_innings2",
"venue_avg_total_runs",
"weather_temp", 
"weather_humidity", 
"weather_dew",
"weather_windspeed", 
"weather_precip"
]

target = "total_fantasy_points"

X_train = train[features]
y_train = train[target]
X_test = test[features]
y_test = test[target]

In [1136]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(24367, 26)
(24367,)
(3542, 26)
(3542,)


In [1137]:
y_test_reset = y_test.reset_index(drop=True)
y_pred = X_test["rolling_avg_fantasy_5"].reset_index(drop=True)
error = abs(y_pred - y_test_reset)
MAE = error.mean()
print(MAE)

23.739356295878036


In [1138]:
from xgboost import XGBRegressor
regressor = XGBRegressor(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
regressor.fit(X_train, y_train)
regressor.predict(X_test)

array([27.749329, 38.642975, 43.60116 , ..., 15.21264 , 18.925034,
       51.27315 ], shape=(3542,), dtype=float32)

In [1139]:
import numpy as np
y_predi = regressor.predict(X_test)
error_X = abs(y_predi - y_test.values)
MAE = np.mean(error_X)
print(MAE)

21.46945881591042


In [1140]:
y_pred_train = regressor.predict(X_train)
error_train = abs(y_pred_train - y_train.values)
MAE_train = np.mean(error_train)
print(f"Train MAE: {MAE_train}")
print(f"Test MAE: {MAE}")

Train MAE: 17.68867466514725
Test MAE: 21.46945881591042


In [1141]:
importance = pd.Series(
    regressor.feature_importances_,
    index=features
).sort_values(ascending=False)
print(importance)

batting_position                  0.367450
career_strike_rate                0.043014
role_encoded                      0.038768
rolling_avg_runs_5                0.038164
rolling_batting_contribution_5    0.030211
rolling_avg_fantasy_10            0.030188
won_toss                          0.027148
venue_avg_innings1                0.027066
venue_avg_total_runs              0.026501
weather_windspeed                 0.025714
weather_humidity                  0.025406
weather_temp                      0.025005
weather_dew                       0.024777
venue_avg_innings2                0.023718
rolling_std_fantasy_10            0.023311
matches_played                    0.022772
weather_precip                    0.022553
venue_std_fantasy                 0.022349
rolling_bowling_contribution_5    0.021853
expanding_season_fantasy_std      0.021640
rolling_avg_wickets_5             0.021342
opposition_std_fantasy            0.019885
rolling_avg_fantasy_5             0.019600
venue_first

In [1142]:
X_train_single = train[["batting_position"]]
X_test_single = test[["batting_position"]]

from lightgbm import LGBMRegressor
lgbm_single = LGBMRegressor(n_estimators=500, max_depth=4, learning_rate=0.05, 
                              subsample=0.8, colsample_bytree=0.8, random_state=42)
lgbm_single.fit(X_train_single, y_train)

preds_single = lgbm_single.predict(X_test_single)
mae_single = np.mean(np.abs(preds_single - y_test.values))
print(f"MAE using ONLY batting_position: {mae_single:.2f}")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001009 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12
[LightGBM] [Info] Number of data points in the train set: 24367, number of used features: 1
[LightGBM] [Info] Start training from score 27.049904
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, b

In [1143]:
from lightgbm import LGBMRegressor

lgbm = LGBMRegressor(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
lgbm.fit(X_train, y_train)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001006 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4693
[LightGBM] [Info] Number of data points in the train set: 24367, number of used features: 26
[LightGBM] [Info] Start training from score 27.049904
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain

,max_depth,4
,learning_rate,0.05
,n_estimators,500
,subsample,0.8
,colsample_bytree,0.8
,random_state,42
,boosting_type,'gbdt'
,num_leaves,31
,subsample_for_bin,200000
,objective,None
,class_weight,None


In [1144]:
y_predi = lgbm.predict(X_test)
error_X = abs(y_predi - y_test.values)
MAE = np.mean(error_X)
print(MAE)

21.125363018885842


In [1145]:
y_pred_train = lgbm.predict(X_train)
MAE_train = np.mean(abs(y_pred_train - y_train.values))
print(f"Train MAE: {MAE_train}")
print(f"Test MAE: {MAE}")

Train MAE: 17.996818955438197
Test MAE: 21.125363018885842


In [1146]:
import pickle
import os

os.makedirs("models", exist_ok=True)

with open("models/lgbm_baseline.pkl", "wb") as f:
    pickle.dump(lgbm, f)

print("Model saved")

Model saved


In [1147]:
os.makedirs("data/processed", exist_ok= True)
df.to_csv("data/processed/player_match_features.csv", index = False)
print(len(df))

27909


In [1148]:
df.columns == "Nan"

array([False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False])

In [1149]:
os.makedirs("data/processed", exist_ok= True)
df.to_csv("data/processed/player_match_features.csv", index = False)
print(len(df))

27909


In [1150]:
unique_weather_queries = df[["date", "city"]].drop_duplicates()
print(len(unique_weather_queries))

1220


In [1151]:
import pickle
with open("models/lgbm_weather.pkl","wb") as f: 
    pickle.dump(lgbm, f)
print("Saved")

Saved


In [1152]:
print(len(features))  # should include the 5 weather columns
print("weather_temp" in features)

26
True
